# 4.0 Explainability plots

---

### What this notebook produces

For a few validation molecules it renders **per-atom attribution maps**: every
atom of the generated SMILES is coloured by how much probability the decoder
assigned to the token(s) that produced it. Green = the model was confident
about that atom, red = it was not. This is the visual counterpart of the
accuracy tables in §1.0: it shows *where* a molecule was built confidently.

The input is always one of the result pickles written by the evaluation
pipeline (`*_exp_data.pkl`, `*_exp_sim_data_*.pkl`), which store, for each target
SMILES, a list of generations.

For every generation the notebook writes, into `experiments/`:

| File | Content |
| --- | --- |
| `molecule_{i}_target.svg` | 2D structure of the target, all atoms at probability 1.0 |
| `molecule_{i}_generation_{n}.svg` | the n-th generated structure, atoms coloured by probability |
| `molecule_{i}_..._label.png` | the text block (generation id, probability, HSQC/COSY error, Tanimoto) |
| `molecule_{i}_..._str.svg` | the SMILES string itself, one coloured cell per character |

### The two "versions" in this notebook

* **cell 1-4** -- first version: character-level probabilities mapped onto atoms
  by a simple SMILES heuristic, label text embedded in the SVG, no canonicalisation.
* **cell 6** -- "version with alignment": canonicalises the SMILES with
  `Chem.CanonSmiles` before drawing, splits the label into a separate PNG, and
  adds the coloured-SMILES strip. Cells 5-6 are an *alternative pipeline*, not an
  extension: they redefine `visualize_molecules` with a different return type.
* **cell 13** -- unrelated leftover: a small RDKit/SVG image demo with three dummy
  molecules ("alignment" here only means canonical SMILES + `Compute2DCoords`).

> **Run order matters.** Cell 6 calls `process_probabilities`, `create_label_image`
> and `get_color` without defining them -- it relies on cells 1 and 4 having run
> first. Each of those redefinitions complains about nothing but silently changes
> the meaning of the earlier functions.

### Issues found while annotating (comments only, code unchanged)

1. **cell 1** `process_probabilities` uses a character heuristic (`char.isalpha()`),
   so two-letter elements (`Cl`, `Br`) and lowercase aromatic atoms are split into
   several "atoms"; the mismatch is then patched by filling with `1.0`.
2. **cell 1** the drawing canvas is `300x300` but the SVG is re-declared as
   `viewBox="0 0 300 500"` with a `500`-tall background -- the bottom label area is
   outside the original canvas.
3. **cell 6** canonicalising the SMILES *reorders* the string, so the per-character
   probabilities no longer correspond to the characters they are zipped with.
4. **cell 6** `generate_colored_molecule` returns `(svg, canonical_smiles)` while
   cell 1/4's version returns just `svg` -- same name, different contract.
5. **cell 6** `generate_noncolored_molecule_svgs` uses an undefined variable
   `idx` (the loop variable is `smile_idx`) and would raise `NameError`.
6. **cell 1/4 vs cell 6** use *different* positional indices into the same generation
   tuple (`[3]`, `[-3]`, `[-2]` vs `[3]`, `[4]`, `[5]`), i.e. the two cells target
   different pickle layouts.
7. **cells 2, 5** still contain absolute cluster/Colab paths; **cell 2** is
   entirely dead (its call is commented out).
8. **cells 7, 9, 14** are intentionally empty placeholders.

In [3]:
# =============================================================================
# CELL 1 -- Core helpers for the explainability figures
#
# Data flow:
#   pickle (*_exp_data.pkl)
#       -> load_data()                  returns results_dict_bl_ZINC
#       -> visualize_molecules()        loops over target molecules / generations
#            -> generate_colored_molecule(smi, prob_list, label)
#                 -> process_probabilities(smi, prob_list)   token probs -> per-atom
#                 -> get_color(prob)                          prob -> RGB
#                 -> RDKit MolDraw2DSVG                       2D structure + highlights
#            -> write one .svg per molecule
#
# What "probability" means here:
#   prob_list[k] is the decoder probability of the k-th *character* of the
#   SMILES string (the sequence the model actually emitted). Because one atom can
#   span several characters ("Cl", "Br", "[nH]") and ring-closure digits are not
#   atoms at all, the character list has to be folded into an atom list before it
#   can be used as an RDKit highlight colour. That folding is
#   process_probabilities().
#
# NOTE (known issue, left unchanged): the folding is a heuristic, not a real
#   tokenizer alignment -- `char.isalpha()` treats every letter as starting a new
#   atom, so "Cl" becomes two entries, "[nH]" becomes three, and lowercase
#   aromatic atoms are counted separately. When the resulting count differs from
#   mol.GetNumAtoms() the list is truncation/padded with 1.0, which can therefore
#   colour the wrong atom (the warning in the console is this path being taken).
#
# NOTE (layout issue, left unchanged): the drawing canvas is 300x300
#   (MolDraw2DSVG(300, 300)) but the SVG header is then rewritten to
#   viewBox="0 0 300 500" with a 500-px-tall white background and label text at
#   y=320..- the label block sits outside the area RDKit drew into. Cell 4 fixes
#   this by creating a 300x500 canvas; this cell was left unchanged on purpose.
# =============================================================================

import pickle
import os
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import IPythonConsole   # enables inline molecule rendering in Jupyter
from rdkit.Chem.Draw import rdMolDraw2D      # low-level SVG/PNG drawing backend

import numpy as np
from functools import reduce                 # reduce(mul, probs) = product of probabilities
import operator
import matplotlib.pyplot as plt
import warnings

def load_data(pkl_path: str) -> dict:
    """Load the evaluation pickle and return its main results dict.

    Args:
        pkl_path: path of a `*_exp_data.pkl` / `*_exp_sim_data_*.pkl` file written
            by the validation pipeline.

    Returns:
        The value stored under the key 'results_dict_bl_ZINC':
        {target_smiles: [ [ [gen_smiles, ..., token_probs, ..., metrics], ... ] ]}
        i.e. target -> list-of-batches -> list-of-generations -> tuple of fields.
        The exact field positions differ between pickles -- see the indexing notes
        in visualize_molecules().
    """
    with open(pkl_path, 'rb') as file:
        data = pickle.load(file)
    return data['results_dict_bl_ZINC']

def process_probabilities(smile: str, probabilities: list) -> list:
    """Fold per-CHARACTER probabilities into per-ATOM probabilities.

    Walks the SMILES string and the aligned probability list simultaneously.
    A letter (or '[' / ']') is treated as the start of a new atom; every other
    character (digits, '(', ')', '=', '#', '-', '.', '@', ...) is a "non-atom"
    symbol belonging to the current atom. Non-atom probabilities are averaged
    into the two neighbouring atoms so that no probability mass is lost.

    Args:
        smile: the SMILES string the probabilities refer to.
        probabilities: one probability per character of `smile`.

    Returns:
        List of floats, one per atom, in the order RDKit assigns atom indices.
        May be shorter/longer than the molecule's atom count (see NOTE in the
        header); the difference is repaired with 1.0 fillers.

    Example (schematic):
        smile = "CCO", probabilities = [0.9, 0.8, 0.7] -> [0.9, 0.8, 0.7]
        smile = "CC(=O)O", probabilities = [0.9,0.8,0.1,0.9,0.6,0.7]
        --> the '=' and '(' probabilities get averaged into neighbours, giving
            three atom values instead of five character values.
    """
    mol = Chem.MolFromSmiles(smile)
    # A malformed SMILES makes atom-level colouring impossible -> fail loudly
    if mol is None:
        raise ValueError("Invalid SMILES string")

    num_atoms = mol.GetNumAtoms()   # ground truth: how many values we need
    atom_probabilities = []         # finished per-atom values
    current_atom_prob = None        # probability of the atom currently being read
    non_atom_probs = []             # probabilities of symbols since that atom started

    for char, prob in zip(smile, probabilities):
        if char.isalpha() or char == '[' or char == ']':  # Start of an atom symbol
            # Close the previous atom before starting a new one
            if current_atom_prob is not None:
                if non_atom_probs:
                    # Average the non-atom probabilities with the current and previous atom
                    # (+2 = one term for the old atom, one for the new atom's prob)
                    avg_prob = (current_atom_prob + sum(non_atom_probs) + prob) / (len(non_atom_probs) + 2)
                    atom_probabilities.append(avg_prob)
                    non_atom_probs = []   # consumed by this atom
                else:
                    # No symbols in between -> the atom keeps its own probability
                    atom_probabilities.append(current_atom_prob)
            current_atom_prob = prob
        else:  # Non-atom character
            # Buffer it; it will be averaged into the neighbouring atoms
            non_atom_probs.append(prob)

    # Handle the last atom
    # (the loop above only flushes an atom when the NEXT atom starts)
    if current_atom_prob is not None:
        if non_atom_probs:
            # Average the non-atom probabilities with the last atom
            avg_prob = (current_atom_prob + sum(non_atom_probs)) / (len(non_atom_probs) + 1)
            atom_probabilities.append(avg_prob)
        else:
            atom_probabilities.append(current_atom_prob)

    # Ensure we have the correct number of probabilities
    # Too many (heuristic over-split) -> keep the first num_atoms values
    if len(atom_probabilities) > num_atoms:
        atom_probabilities = atom_probabilities[:num_atoms]
    # Too few -> pad with 1.0 ("fully confident") so RDKit can still colour every atom.
    # NOTE: the padding also hides real misalignment, hence the explicit warning.
    if len(atom_probabilities) != num_atoms:
        warnings.warn(f"Number of processed probabilities ({len(atom_probabilities)}) "
                      f"does not match number of atoms in molecule ({num_atoms}). "
                      "This may lead to incorrect coloring. "
                      "For now probabilities of 1 are added to keep the code working")
        atom_probabilities.extend([1.0] * (num_atoms - len(atom_probabilities)))

    return atom_probabilities

def get_color(prob: float) -> tuple:
    """Map a probability in [0, 1] to an RGB triple in [0, 1].

    The mapping is a red->green ramp with a constant blue component:
        prob = 0.0 -> red   = 1.00, green = 0.00, blue = 0.70  (magenta/red)
        prob = 0.5 -> red   = 0.80, green = 0.45, blue = 0.70
        prob = 1.0 -> red   = 0.60, green = 0.90, blue = 0.70  (light green)
    i.e. high decoder confidence is green, low confidence is red. The blue
    constant keeps the palette pastel so black atom labels stay readable.

    Args:
        prob: decoder probability assigned to this atom.

    Returns:
        (r, g, b) floats in [0, 1] (RDKit's highlight colour format).
    """
    green_value = prob * 0.9
    red_value = 1.0 - prob * 0.40
    blue_value = 0.7
    # Multiply by 255 and divide again -> quantise to 8-bit steps, then return 0-1
    return (float((red_value*255)/255), float((green_value*255)/255), float((blue_value*255)/255))
import xml.etree.ElementTree as ET
import re


def visualize_molecules(pkl_path: str, output_folder: str, num_molecules: int = 5, generations_per_molecule: int = 3) -> None:
    """Write one target SVG plus N generation SVGs per molecule.

    Args:
        pkl_path: result pickle to read (see load_data).
        output_folder: destination folder (created if missing).
        num_molecules: how many targets to render (clipped to the number stored).
        generations_per_molecule: how many generations per target.

    Generation tuple layout used by THIS cell:
        [0]  generated SMILES
        [3]  per-token probabilities (torch tensor -> .tolist())
        [-3] Tanimoto similarity to the target
        [-2] [HSQC error, COSY error]
    NOTE: cell 6 indexes the very same structure as [0], [3], [4], [5]; the two
    cells therefore expect different pickle versions.
    """
    result = load_data(pkl_path)
    keys_list = list(result.keys())   # target SMILES, in insertion order

    os.makedirs(output_folder, exist_ok=True)

    # min(...) protects against num_molecules > number of available targets
    for index in range(min(num_molecules, len(keys_list))):
        target_smi = keys_list[index]

        # Target molecule
        # All atoms get probability 1.0 -> uniform green, i.e. "everything is known"
        smi = target_smi
        prob_list = [1 for _ in smi]
        label = "Target"
        svg = generate_colored_molecule(smi, prob_list, label)

        with open(os.path.join(output_folder, f"molecule_{index}_target.svg"), "w", encoding='utf-8') as f:
            f.write(svg)

        # Generated molecules
        # i-1 because the stored generations are 0-based while i is 1-based
        for i in range(1, generations_per_molecule + 1):
            smi = result[target_smi][0][i-1][0]
            prob_list = result[target_smi][0][i-1][3].tolist()
            HSQC_err = result[target_smi][0][i-1][-2][0]
            COSY_err = result[target_smi][0][i-1][-2][1]
            tani_sim = result[target_smi][0][i-1][-3]
            # Joint probability of the whole sequence = product of token probabilities
            # (log-space would be numerically safer, but this is only a display number)
            prob_product = reduce(operator.mul, prob_list, 1)

            label = f"Generation: {i}\nProbability: {prob_product:.2e}\nHSQC Error: {HSQC_err:.3f}\nCOSY Error: {COSY_err:.3f}\nTanimoto Sim: {tani_sim:.3f}"

            svg = generate_colored_molecule(smi, prob_list, label)

            with open(os.path.join(output_folder, f"molecule_{index}_generation_{i}.svg"), "w", encoding='utf-8') as f:
                f.write(svg)

    # NOTE: the message always reports num_molecules, even when the loop ran fewer
    # iterations because the pickle held fewer targets.
    print(f"Generated SVGs for {num_molecules} molecules with {generations_per_molecule} generations each.")


import re

def generate_colored_molecule(smile: str, probabilities: list, label: str) -> str:
    """Render a 2D depiction with per-atom confidence colours and a text label.

    Args:
        smile: molecule to draw.
        probabilities: per-character decoder probabilities (folded to atoms inside).
        label: multi-line text drawn under the structure ("\\n" separated).

    Returns:
        SVG document as a string: RDKit drawing + white background + label tspans.
    """
    molecule = Chem.MolFromSmiles(smile)
    # Fold character-level probabilities into the atom order RDKit uses
    atom_probabilities = process_probabilities(smile, probabilities)

    # Increase the SVG size more significantly
    d = rdMolDraw2D.MolDraw2DSVG(300, 300)  # Keep molecule size at 300x300
    # Black/white atom palette: the highlight colours stay the only colour cue
    d.drawOptions().useBWAtomPalette()
    # {atom_index: rgb} -- one entry per atom
    highlight_atom_colors = {i: get_color(prob) for i, prob in enumerate(atom_probabilities)}

    d.DrawMolecule(molecule, highlightAtoms=list(highlight_atom_colors.keys()), highlightAtomColors=highlight_atom_colors)
    d.FinishDrawing()

    svg = d.GetDrawingText()

    # Parse the SVG
    # (RDKit output is usually well-formed; the fallback covers exotic molecule
    # names breaking the XML escaping)
    try:
        root = ET.fromstring(svg)
    except ET.ParseError:
        # If parsing fails, try to clean up the SVG
        svg = re.sub(r'</?svg[^>]*>', '', svg)  # Remove svg tags
        svg = f'<svg width="300" height="500">{svg}</svg>'  # Wrap in new svg tags with increased height
        root = ET.fromstring(svg)

    # Set or adjust the viewBox
    # NOTE: the canvas was created as 300x300, so 300x500 stretches the drawing.
    root.set('viewBox', '0 0 300 500')  # Set viewBox to match new dimensions
    root.set('width', '300')
    root.set('height', '500')

    # Add a white background rectangle
    background = ET.SubElement(root, 'rect', {
        'width': '300',
        'height': '500',
        'fill': 'white'
    })

    # Move the background to the back
    # (SubElement appended it last; re-inserting at position 0 puts it behind the drawing)
    root.insert(0, background)

    # Add label
    text_element = ET.SubElement(root, 'text', {
        'x': '10',
        'y': '320',  # Start labels below the molecule
        'font-family': 'sans-serif',
        'font-size': '14px',  # Increased font size
        'fill': 'black'
    })

    # One <tspan> per line; y advances by 20 px each line
    y = 320
    for line in label.split('\n'):
        tspan = ET.SubElement(text_element, 'tspan', {
            'x': '10',
            'y': str(y)
        })
        tspan.text = line
        y += 20  # Increased line spacing

    # Convert back to string
    # (encoding='unicode' avoids the 'ns0:' namespace prefixes ET would add otherwise)
    svg_str = ET.tostring(root, encoding='unicode')

    return svg_str

In [11]:
# =============================================================================
# CELL 2 -- Dead configuration for the original cluster run
#
# This cell only assigns two constants; the actual rendering call is commented out,
# so running it does nothing except define `pkl_path` and `output_folder`.
#
# NOTE (known issue, left unchanged): both paths are absolute HPC/Colab locations
#   ('/projects/cc/se_users/...'), so they do not exist on this machine. The
#   portable equivalent is what cell 4 uses:
#       base_dir = os.getcwd()
#       pkl_path = os.path.join(base_dir, "MMST_website/pkl_files/<exp>.pkl")
# =============================================================================

# Call this function to display the generated SVGs in your notebook
#display_molecules(output_folder)
pkl_path = '/projects/cc/se_users/knlr326/1_NMR_project/2_Notebooks/nmr_project/1_Dataexploration/2_paper_code/Experiments_SLURM/20.0_SLURM_MasterTransformer/_ISAK/Runfolder/pkl_files/20240731_223447_IC_exp_data.pkl'
output_folder = '/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/___FIGURES_PAPERS/Figures_Paper_2/precomputed_raw_data/20241022_Explainability_Model'
#visualize_molecules(pkl_path, output_folder)

Generated SVGs for 5 molecules with 3 generations each.


In [4]:
# =============================================================================
# CELL 3 -- Display helper for the SVGs written by visualize_molecules
#
# `IPython.display.SVG(filename=...)` inlines the file into the notebook output.
# Nothing is recomputed, so this is a cheap way to check the figures after a run.
#
# NOTE (known issue, left unchanged): the loop count comes from the arguments,
#   not from the files actually present -- asking for more molecules/generations
#   than were rendered raises FileNotFoundError instead of stopping early.
#
# NOTE (dependency, left unchanged): `os` is imported in cells 1/4, so this cell
#   only works after one of them has run (in a fresh kernel it is a NameError).
# =============================================================================

import xml.etree.ElementTree as ET
import re
# To display SVGs in the notebook
from IPython.display import SVG, display

def display_molecules(output_folder: str, num_molecules: int = 5, generations_per_molecule: int = 3) -> None:
    """Inline-display the target + generation SVGs for the first `num_molecules` targets.

    Args:
        output_folder: folder written by visualize_molecules().
        num_molecules: how many targets to show (must exist on disk).
        generations_per_molecule: how many generations per target (must exist).
    """
    for index in range(num_molecules):
        # Target structure first ...
        display(SVG(filename=os.path.join(output_folder, f"molecule_{index}_target.svg")))
        # ... then every generation, in order
        for i in range(1, generations_per_molecule + 1):
            display(SVG(filename=os.path.join(output_folder, f"molecule_{index}_generation_{i}.svg")))

In [6]:
# =============================================================================
# CELL 4 -- Full version: structure + separate label PNG + coloured SMILES strip
#
# This cell is self-contained (it re-defines load_data / process_probabilities /
# get_color from cell 1) and extends the output of cell 1 with two extra artefacts
# per molecule:
#
#   molecule_<i>_<kind>.svg              the 2D depiction, atoms coloured by prob
#   molecule_<i>_<kind>_label.png        the text block as a raster image
#                                        (generation, joint probability, HSQC/COSY
#                                        error, Tanimoto) -- drawn with PIL so the
#                                        figure tool the paper uses can place it
#                                        independently of the SVG
#   molecule_<i>_<kind>_str.svg          every character of the SMILES in its own
#                                        coloured box ("sequence confidence strip")
#
# When it runs (bottom of the cell) it writes into <repo>/experiments -- note this
# is the experiments *root*, not experiments/exp_1.
#
# NOTE (known issue, left unchanged): the label string is only used for the PNG.
#   generate_colored_molecule() is always called with label="" here, so the SVG
#   keeps an empty <text> element. The empty element is harmless but it means the
#   SVG is not a complete figure on its own -- it must be composited with the PNG.
#
# NOTE (known issue, left unchanged): `create_label_image` uses a bare `except:`,
#   which also swallows KeyboardInterrupt and programming errors. The fallback is
#   PIL's bitmap default font, whose size differs a lot from DejaVu Sans 14px.
# =============================================================================

import pickle
import os
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem.Draw import rdMolDraw2D

import numpy as np
from functools import reduce
import operator
import matplotlib.pyplot as plt
import warnings

def load_data(pkl_path: str) -> dict:
    """Load 'results_dict_bl_ZINC' from the evaluation pickle (same as cell 1).

    Args:
        pkl_path: path of a result pickle.

    Returns:
        dict target_smiles -> list of batches -> list of generation tuples.
    """
    with open(pkl_path, 'rb') as file:
        data = pickle.load(file)
    return data['results_dict_bl_ZINC']

def process_probabilities(smile: str, probabilities: list) -> list:
    """Fold per-character probabilities into per-atom probabilities.

    Identical to cell 1; see the header there for the heuristic and its caveats.
    Short version: letters start a new atom, everything else is averaged into the
    neighbouring atoms, and the result is truncated/padded to mol.GetNumAtoms().
    """
    mol = Chem.MolFromSmiles(smile)
    if mol is None:
        raise ValueError("Invalid SMILES string")

    num_atoms = mol.GetNumAtoms()
    atom_probabilities = []
    current_atom_prob = None
    non_atom_probs = []

    for char, prob in zip(smile, probabilities):
        if char.isalpha() or char == '[' or char == ']':  # Start of an atom symbol
            if current_atom_prob is not None:
                if non_atom_probs:
                    # Average the non-atom probabilities with the current and previous atom
                    avg_prob = (current_atom_prob + sum(non_atom_probs) + prob) / (len(non_atom_probs) + 2)
                    atom_probabilities.append(avg_prob)
                    non_atom_probs = []
                else:
                    atom_probabilities.append(current_atom_prob)
            current_atom_prob = prob
        else:  # Non-atom character
            non_atom_probs.append(prob)

    # Handle the last atom
    if current_atom_prob is not None:
        if non_atom_probs:
            # Average the non-atom probabilities with the last atom
            avg_prob = (current_atom_prob + sum(non_atom_probs)) / (len(non_atom_probs) + 1)
            atom_probabilities.append(avg_prob)
        else:
            atom_probabilities.append(current_atom_prob)

    # Ensure we have the correct number of probabilities
    if len(atom_probabilities) > num_atoms:
        atom_probabilities = atom_probabilities[:num_atoms]
    if len(atom_probabilities) != num_atoms:
        warnings.warn(f"Number of processed probabilities ({len(atom_probabilities)}) "
                      f"does not match number of atoms in molecule ({num_atoms}). "
                      "This may lead to incorrect coloring. "
                      "For now probabilities of 1 are added to keep the code working")
        atom_probabilities.extend([1.0] * (num_atoms - len(atom_probabilities)))

    return atom_probabilities

def get_color(prob: float) -> tuple:
    """Probability -> pastel RGB (green = confident, red = uncertain); same as cell 1."""
    green_value = prob * 0.9
    red_value = 1.0 - prob * 0.40
    blue_value = 0.7
    return (float((red_value*255)/255), float((green_value*255)/255), float((blue_value*255)/255))

import xml.etree.ElementTree as ET
import re

import matplotlib.pyplot as plt
# PIL is used only to rasterise the multi-line label text into a separate PNG
from PIL import Image, ImageDraw, ImageFont
from PIL import Image, ImageDraw, ImageFont   # duplicate import (harmless)
import matplotlib.font_manager as fm           # used to locate a real TTF font

def create_label_image(label: str, width: int = 300, height: int = 120):
    """Render a multi-line label string into a white PIL image.

    Args:
        label: text with '\\n' line separators (5 lines for a generation).
        width: image width in px (matches the SVG width so they stack cleanly).
        height: image height in px; 120 fits 5 lines at 24 px spacing.

    Returns:
        PIL.Image (RGB, white background, black text).
    """
    # Create a white image
    image = Image.new('RGB', (width, height), color='white')
    draw = ImageDraw.Draw(image)

    # Try to use a common sans-serif font
    try:
        # Look for Arial or Helvetica
        # (findfont falls back through fontconfig; the actual file may be DejaVu)
        font_path = fm.findfont(fm.FontProperties(family='DejaVu Sans'))
        font = ImageFont.truetype(font_path, 14)  # Increased font size to 14
    except:
        # If not found, use default
        font = ImageFont.load_default()

    # Draw the text
    y_text = 5
    for line in label.split('\n'):
        draw.text((5, y_text), line, fill='black', font=font)
        y_text += 24  # Increased line spacing

    return image


def html_to_svg(html_string: str, width: int = 600, height: int = 30) -> str:
    """Convert the coloured <span> string produced by generate_colored_html into an SVG.

    Each coloured span becomes a 20x20 <rect> plus one <text> character on top,
    laid out left-to-right at a fixed 20 px pitch -- i.e. a monospace-style
    "confidence strip" for the whole SMILES.

    Args:
        html_string: output of generate_colored_html().
        width: total SVG width (callers pass len(smiles)*20 + 10).
        height: total SVG height (one row of boxes).

    Returns:
        SVG document as a string.
    """
    # Extract the colored spans from the HTML string
    # (colour + the single character it wraps; the regex assumes exactly this
    #  attribute order, which is why generate_colored_html is written the same way)
    spans = re.findall(r'<span style="background-color: ([^;]+);[^>]+>([^<]+)</span>', html_string)

    # Create the SVG string
    svg = f'<svg width="{width}" height="{height}" xmlns="http://www.w3.org/2000/svg">'
    svg += '<style>text { font-family: monospace; font-size: 20px; }</style>'

    x = 5  # Starting x position
    for color, char in spans:
        # Coloured cell behind the character
        svg += f'<rect x="{x}" y="5" width="20" height="20" fill="{color}"/>'
        # Character on top (offset by 5 px so it is centred in the cell)
        svg += f'<text x="{x+5}" y="22" fill="black">{char}</text>'
        x += 20  # Move to the next character position

    svg += '</svg>'
    return svg

def generate_colored_html(smile: str, probabilities: list) -> str:
    """Build one coloured <span> per SMILES character (the HTML stage before SVG).

    Args:
        smile: SMILES string.
        probabilities: one probability per character (same alignment assumptions
            as process_probabilities, but here no folding happens).

    Returns:
        A <span> block with `display:inline-block` cells, one per character.
        NOTE: a character that is whitespace or '<' would break the HTML; SMILES
        never contains those, so no escaping is done.
    """
    html_string = '<span style="font-family: monospace;">'

    for char, prob in zip(smile, probabilities):
        color = get_color(prob)
        # get_color returns 0-1 floats -> scale back to 0-255 for CSS
        css_color = f'rgb({int(color[0]*255)}, {int(color[1]*255)}, {int(color[2]*255)})'
        html_string += f'<span style="background-color: {css_color}; padding: 1px; display: inline-block;">{char}</span>'

    html_string += '</span>'
    return html_string

def visualize_molecules(pkl_path: str, output_folder: str, num_molecules: int = 5, generations_per_molecule: int = 3) -> None:
    """Write SVG + label PNG + coloured-SMILES SVG for each target and generation.

    Args:
        pkl_path: result pickle (see load_data).
        output_folder: destination directory (created if missing).
        num_molecules: number of targets, clipped to those stored in the pickle.
        generations_per_molecule: generations per target; indexes [0]..[n-1] of
            the stored generation list.

    Generation tuple layout used by THIS cell:
        [0] generated SMILES, [3] token probabilities,
        [-3] Tanimoto, [-2] [HSQC error, COSY error].
        (cell 6 uses [4] / [5] instead -- different pickle layout.)
    """
    result = load_data(pkl_path)
    keys_list = list(result.keys())

    os.makedirs(output_folder, exist_ok=True)

    for index in range(min(num_molecules, len(keys_list))):
        target_smi = keys_list[index]

        # Target molecule
        smi = target_smi
        # Every character gets probability 1.0 -> uniform "fully known" colour
        prob_list = [1 for _ in smi]
        label = "Target"
        # Empty label inside the SVG; the text is written to a PNG instead
        svg = generate_colored_molecule(smi, prob_list, "")  # Empty label for SVG

        # Three artefacts share the same stem, differing only in suffix
        svg_filename = os.path.join(output_folder, f"molecule_{index}_target.svg")
        png_filename = os.path.join(output_folder, f"molecule_{index}_target_label.png")
        str_svg_filename = os.path.join(output_folder, f"molecule_{index}_target_str.svg")

        with open(svg_filename, "w", encoding='utf-8') as f:
            f.write(svg)

        # Create and save label image
        label_image = create_label_image(label)
        label_image.save(png_filename)

        # Generate and save colored SMILES string as SVG
        colored_html = generate_colored_html(smi, prob_list)
        colored_svg = html_to_svg(colored_html, width=len(smi)*20+10)  # Adjust width based on SMILES length
        with open(str_svg_filename, "w", encoding='utf-8') as f:
            f.write(colored_svg)

        # Generated molecules
        for i in range(1, generations_per_molecule + 1):
            # Stored generations are 0-based, `i` in the file names is 1-based
            smi = result[target_smi][0][i-1][0]
            prob_list = result[target_smi][0][i-1][3].tolist()
            HSQC_err = result[target_smi][0][i-1][-2][0]
            COSY_err = result[target_smi][0][i-1][-2][1]
            tani_sim = result[target_smi][0][i-1][-3]
            # Joint sequence probability = product of the token probabilities
            prob_product = reduce(operator.mul, prob_list, 1)

            label = f"Generation: {i}\nProbability: {prob_product:.2e}\nHSQC Error: {HSQC_err:.3f}\nCOSY Error: {COSY_err:.3f}\nTanimoto Sim: {tani_sim:.3f}"

            svg = generate_colored_molecule(smi, prob_list, "")  # Empty label for SVG

            svg_filename = os.path.join(output_folder, f"molecule_{index}_generation_{i}.svg")
            png_filename = os.path.join(output_folder, f"molecule_{index}_generation_{i}_label.png")
            str_svg_filename = os.path.join(output_folder, f"molecule_{index}_generation_{i}_str.svg")

            with open(svg_filename, "w", encoding='utf-8') as f:
                f.write(svg)

            # Create and save label image
            label_image = create_label_image(label)
            label_image.save(png_filename)

            # Generate and save colored SMILES string as SVG
            colored_html = generate_colored_html(smi, prob_list)
            colored_svg = html_to_svg(colored_html, width=len(smi)*20+10)  # Adjust width based on SMILES length
            with open(str_svg_filename, "w", encoding='utf-8') as f:
                f.write(colored_svg)

    print(f"Generated SVGs, label PNGs, and colored SMILES SVGs for {num_molecules} molecules with {generations_per_molecule} generations each.")


def generate_colored_molecule(smile: str, probabilities: list, label: str) -> str:
    """Draw a 300x500 SVG: atom highlights + (usually empty) label text block.

    Args:
        smile: molecule to draw.
        probabilities: per-character decoder probabilities.
        label: text drawn under the structure; passed as "" by visualize_molecules
            because the label is exported as a separate PNG.

    Returns:
        SVG string.
    """
    molecule = Chem.MolFromSmiles(smile)
    atom_probabilities = process_probabilities(smile, probabilities)

    # Increase the SVG size more significantly
    # (300x500 canvas here, unlike cell 1's 300x300 -- this is the version that
    #  actually leaves room for the label block)
    d = rdMolDraw2D.MolDraw2DSVG(300, 500)  # Increase height to accommodate labels
    d.drawOptions().useBWAtomPalette()
    highlight_atom_colors = {i: get_color(prob) for i, prob in enumerate(atom_probabilities)}

    d.DrawMolecule(molecule, highlightAtoms=list(highlight_atom_colors.keys()), highlightAtomColors=highlight_atom_colors)
    d.FinishDrawing()

    svg = d.GetDrawingText()

    # Parse the SVG
    try:
        root = ET.fromstring(svg)
    except ET.ParseError:
        # If parsing fails, try to clean up the SVG
        svg = re.sub(r'</?svg[^>]*>', '', svg)  # Remove svg tags
        svg = f'<svg width="300" height="500">{svg}</svg>'  # Wrap in new svg tags with increased height
        root = ET.fromstring(svg)

    # Set or adjust the viewBox
    root.set('viewBox', '0 0 300 500')  # Set viewBox to match new dimensions
    root.set('width', '300')
    root.set('height', '500')

    # Add a white background rectangle
    background = ET.SubElement(root, 'rect', {
        'width': '300',
        'height': '500',
        'fill': 'white'
    })

    # Move the background to the back
    root.insert(0, background)

    # Add label
    text_element = ET.SubElement(root, 'text', {
        'x': '10',
        'y': '320',  # Start labels below the molecule
        'font-family': 'sans-serif',
        'font-size': '14px',  # Increased font size
        'fill': 'black'
    })

    # Label lines start lower here (y=340) than in cell 1 (y=320)
    y = 340  # Start labels a bit lower
    for line in label.split('\n'):
        tspan = ET.SubElement(text_element, 'tspan', {
            'x': '10',
            'y': str(y)
        })
        tspan.text = line
        y += 20  # Increased line spacing

    # Convert back to string
    svg_str = ET.tostring(root, encoding='unicode')

    return svg_str

# Call this function to display the generated SVGs in your notebook
# display_molecules(output_folder)
base_dir = os.getcwd()
# Demo pickle shipped with the website assets (a single experiment result)
pkl_path = os.path.join(base_dir,"MMST_website/pkl_files/20250703_081012_IC_exp_data.pkl")
# Note: writes into <repo>/experiments, not into experiments/exp_1
output_folder = os.path.join(base_dir,"experiments")

# Side effect on execution: renders 5 targets x 3 generations and writes all files
visualize_molecules(pkl_path, output_folder)

Generated SVGs, label PNGs, and colored SMILES SVGs for 5 molecules with 3 generations each.


### Version with alignment 
- Working - just replace for pkl path from the following folder for the exp, ACD and Sim data

- /projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/___FIGURES_PAPERS/Figures_Paper_2/precomputed_raw_data/20240724_IC_batched_MMT_a1

---

**What "alignment" means here:** before drawing, the generated SMILES is replaced by
`Chem.CanonSmiles(...)` and the 2D coordinates are computed for that canonical molecule,
so the *same* target and its generations share one depiction convention (atom order,
orientation) instead of each being laid out independently. That is what makes the
structures comparable side by side in the paper figure.

**Three caveats worth knowing before you reuse this cell**

1. Canonicalisation rewrites the string, so the per-character probabilities are zipped
   against a *different* sequence than the one the decoder produced. Cell 6 therefore
   colours atoms whose character positions no longer correspond to the tokens.
   The correct fix is to keep the original token alignment and map atom indices through
   `Chem.MolFromSmiles(...).GetAtomWithIdx(...)`, not through string positions.
2. Cell 6 reuses `process_probabilities`, `create_label_image` and `get_color` from
   cells 1/4 without defining them -- **run cell 4 first**, then cell 6.
3. The bullet path above is a cluster path (`/projects/cc/...`). Replace it with a
   repository-relative path such as
   `past_experiments/ChemXriv/8.0_Experiment_Sim_Exp/20240724_IC_batched_MMT_a1/`.
   Cell 6 does exactly that at the bottom of the cell.

The `exp`, `ACD` and `Sim` variants mentioned above are three *pickle families*
(experimental spectra, ACD-predicted spectra, simulated spectra). The code is identical;
only the input file changes, which is why cell 6 hard-codes one file and expects you to
swap `pkl_path` for the other two.

In [30]:
# =============================================================================
# CELL 6 -- "Alignment" version: canonical SMILES + separate labels
#
# Differences from cell 4:
#   * generate_colored_molecule() canonicalises the SMILES and returns
#     (svg, canonical_smiles) instead of just svg -- a DIFFERENT contract under
#     the SAME function name.
#   * explicit rdDepictor.Compute2DCoords(molecule) before drawing.
#   * visualize_molecules() defaults to 8 generations per molecule (cell 4: 3).
#   * generation tuple is read as [0]=SMILES, [3]=probs, [4]=Tanimoto, [5]=[HSQC,COSY]
#     whereas cells 1/4 use [-3] and [-2] -> this cell expects a different pickle.
#   * generate_noncolored_molecule_svgs() added: draws targets without highlights
#     from the key 'results_dict_ZINC_greedy_bl' (note: a *different* dict key).
#
# RUN ORDER: this cell defines neither process_probabilities() nor
# create_label_image() nor load_data(). They must come from cells 1/4, otherwise
# NameError at the first call. The literal comment "... [Keep all the existing
# import statements and helper functions] ..." marks where they would have been.
#
# NOTE (known issue, left unchanged): the canonical SMILES is longer/shorter/differently
#   ordered than the decoded string, so process_probabilities(canonical_smiles,
#   probabilities) zips probabilities against the wrong characters. The colours
#   are therefore only trustworthy when the molecule has no reordering/shortening
#   (e.g. simple acyclic targets). This is the fundamental flaw of the "alignment"
#   approach implemented here.
#
# NOTE (known issue, left unchanged): in generate_noncolored_molecule_svgs the loop
#   variable is `smile_idx` but the file name uses `idx`, which is undefined ->
#   NameError on the first iteration. The final `print(output_path)` outside the
#   loop would fail too if the SMILES list were empty. The `idx` name is a leftover
#   from an earlier version whose loop was `for idx, smiles in enumerate(...)`.
# =============================================================================

import pickle
import os
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem.Draw import rdMolDraw2D

import numpy as np
from functools import reduce
import operator
import matplotlib.pyplot as plt
import warnings
import xml.etree.ElementTree as ET
import re

# ... [Keep all the existing import statements and helper functions] ...
# (this is where load_data / process_probabilities / create_label_image would go;
#  in practice they are taken from cells 1 and 4)

def get_color(prob: float) -> tuple:
    """Probability -> pastel RGB. Identical to cells 1/4.

    prob 0.0 -> red/magenta, prob 1.0 -> light green; constant blue 0.7.
    """
    green_value = prob * 0.9
    red_value = 1.0 - prob * 0.40
    blue_value = 0.7
    return (float((red_value*255)/255), float((green_value*255)/255), float((blue_value*255)/255))
import xml.etree.ElementTree as ET
import re


def generate_colored_molecule(smile: str, probabilities: list, label: str):
    """Draw a canonicalised molecule and RETURN (svg, canonical_smiles).

    Args:
        smile: decoded SMILES.
        probabilities: per-character decoder probabilities (for `smile`, not for
            the canonical string -- see the NOTE in the header).
        label: text for the SVG block; callers pass "" and export the label as PNG.

    Returns:
        Tuple (svg_string, canonical_smiles). NOTE the tuple return: any caller
        written for cell 1/4 (which returns a bare string) will break.
    """
    # Canonicalize SMILES
    # `mol` is parsed but never used -- it is a leftover of an earlier draft
    mol = Chem.MolFromSmiles(smile)
    canonical_smiles = Chem.CanonSmiles(smile)
    molecule = Chem.MolFromSmiles(canonical_smiles)

    # Recompute probabilities for the canonical SMILES
    # (this is the "alignment" step; it assumes character positions still match,
    #  which is only true for molecules whose canonical form equals the decoded form)
    canonical_probabilities = process_probabilities(canonical_smiles, probabilities)

    # Generate 2D coordinates
    # Deterministic 2D layout for the canonical molecule -> comparable depictions
    rdDepictor.Compute2DCoords(molecule)

    # Increase the SVG size more significantly
    d = rdMolDraw2D.MolDraw2DSVG(300, 500)  # Increase height to accommodate labels
    d.drawOptions().useBWAtomPalette()
    highlight_atom_colors = {i: get_color(prob) for i, prob in enumerate(canonical_probabilities)}

    d.DrawMolecule(molecule, highlightAtoms=list(highlight_atom_colors.keys()), highlightAtomColors=highlight_atom_colors)
    d.FinishDrawing()

    svg = d.GetDrawingText()

    # Parse the SVG
    try:
        root = ET.fromstring(svg)
    except ET.ParseError:
        # If parsing fails, try to clean up the SVG
        svg = re.sub(r'</?svg[^>]*>', '', svg)  # Remove svg tags
        svg = f'<svg width="300" height="500">{svg}</svg>'  # Wrap in new svg tags with increased height
        root = ET.fromstring(svg)

    # Set or adjust the viewBox
    root.set('viewBox', '0 0 300 500')  # Set viewBox to match new dimensions
    root.set('width', '300')
    root.set('height', '500')

    # Add a white background rectangle
    background = ET.SubElement(root, 'rect', {
        'width': '300',
        'height': '500',
        'fill': 'white'
    })

    # Move the background to the back
    root.insert(0, background)

    # Add label
    text_element = ET.SubElement(root, 'text', {
        'x': '10',
        'y': '320',  # Start labels below the molecule
        'font-family': 'sans-serif',
        'font-size': '14px',  # Increased font size
        'fill': 'black'
    })

    y = 340  # Start labels a bit lower
    for line in label.split('\n'):
        tspan = ET.SubElement(text_element, 'tspan', {
            'x': '10',
            'y': str(y)
        })
        tspan.text = line
        y += 20  # Increased line spacing

    # Convert back to string
    svg_str = ET.tostring(root, encoding='unicode')

    # NOTE: second return value -- see the docstring
    return svg_str, canonical_smiles


def generate_colored_html(smile: str, probabilities: list) -> str:
    """One coloured <span> per character (identical to cell 4)."""
    html_string = '<span style="font-family: monospace;">'

    for char, prob in zip(smile, probabilities):
        color = get_color(prob)
        css_color = f'rgb({int(color[0]*255)}, {int(color[1]*255)}, {int(color[2]*255)})'
        html_string += f'<span style="background-color: {css_color}; padding: 1px; display: inline-block;">{char}</span>'

    html_string += '</span>'
    return html_string



def html_to_svg(html_string: str, width: int = 600, height: int = 30) -> str:
    """Coloured <span> cells -> SVG boxes (identical to cell 4).

    The regex requires exactly "background-color: X;...>CHAR</span" per cell.
    """
    # Extract the colored spans from the HTML string
    spans = re.findall(r'<span style="background-color: ([^;]+);[^>]+>([^<]+)</span>', html_string)

    # Create the SVG string
    svg = f'<svg width="{width}" height="{height}" xmlns="http://www.w3.org/2000/svg">'
    svg += '<style>text { font-family: monospace; font-size: 20px; }</style>'

    x = 5  # Starting x position
    for color, char in spans:
        svg += f'<rect x="{x}" y="5" width="20" height="20" fill="{color}"/>'
        svg += f'<text x="{x+5}" y="22" fill="black">{char}</text>'
        x += 20  # Move to the next character position

    svg += '</svg>'
    return svg

def visualize_molecules(pkl_path: str, output_folder: str, num_molecules: int = 5, generations_per_molecule: int = 8) -> None:
    """Alignment version: canonical structures + label PNG + coloured SMILES strip.

    Args:
        pkl_path: result pickle (load_data, from cell 1/4).
        output_folder: destination directory.
        num_molecules: number of targets (clipped to what the pickle stores).
        generations_per_molecule: default 8 here (cell 4 used 3) -- the pickles
            this cell targets contain more generations per target.

    Side effects: writes `molecule_<i>_<kind>.svg`, `..._label.png` and `..._str.svg`
    for every target and generation.

    NOTE (dependency): create_label_image() is NOT defined in this cell -> cell 4
    must have been executed (or the function pasted in).
    """
    result = load_data(pkl_path)
    keys_list = list(result.keys())

    os.makedirs(output_folder, exist_ok=True)

    for index in range(min(num_molecules, len(keys_list))):
        target_smi = keys_list[index]

        # Target molecule
        smi = target_smi
        # Probability 1.0 for every character of the (original) target SMILES
        prob_list = [1 for _ in smi]
        label = "Target"
        #import IPython; IPython.embed();

        # Tuple unpacking because this cell's generate_colored_molecule returns two values
        svg, canonical_smi = generate_colored_molecule(smi, prob_list, "")  # Empty label for SVG

        svg_filename = os.path.join(output_folder, f"molecule_{index}_target.svg")
        png_filename = os.path.join(output_folder, f"molecule_{index}_target_label.png")
        str_svg_filename = os.path.join(output_folder, f"molecule_{index}_target_str.svg")

        with open(svg_filename, "w", encoding='utf-8') as f:
            f.write(svg)

        # Create and save label image
        label_image = create_label_image(label)
        label_image.save(png_filename)

        # Generate and save colored SMILES string as SVG
        # NOTE: width is based on the canonical SMILES, which is what gets drawn

        colored_html = generate_colored_html(canonical_smi, prob_list)
        colored_svg = html_to_svg(colored_html, width=len(canonical_smi)*20+10)  # Adjust width based on SMILES length
        with open(str_svg_filename, "w", encoding='utf-8') as f:
            f.write(colored_svg)

        # Generated molecules
        for i in range(1, generations_per_molecule + 1):
            smi = result[target_smi][0][i-1][0]
            #import IPython; IPython.embed();

            prob_list = result[target_smi][0][i-1][3].tolist()
            # Index layout of THIS pickle family: [4]=Tanimoto, [5]=[HSQC, COSY]
            HSQC_err = result[target_smi][0][i-1][5][0]
            COSY_err = result[target_smi][0][i-1][5][1]
            tani_sim = result[target_smi][0][i-1][4]
            prob_product = reduce(operator.mul, prob_list, 1)

            label = f"Generation: {i}\nProbability: {prob_product:.2e}\nHSQC Error: {HSQC_err:.3f}\nCOSY Error: {COSY_err:.3f}\nTanimoto Sim: {tani_sim:.3f}"

            svg, canonical_smi = generate_colored_molecule(smi, prob_list, "")  # Empty label for SVG

            svg_filename = os.path.join(output_folder, f"molecule_{index}_generation_{i}.svg")
            png_filename = os.path.join(output_folder, f"molecule_{index}_generation_{i}_label.png")
            str_svg_filename = os.path.join(output_folder, f"molecule_{index}_generation_{i}_str.svg")

            with open(svg_filename, "w", encoding='utf-8') as f:
                f.write(svg)

            # Create and save label image
            label_image = create_label_image(label)
            label_image.save(png_filename)

            # Generate and save colored SMILES string as SVG
            colored_html = generate_colored_html(canonical_smi, prob_list)
            colored_svg = html_to_svg(colored_html, width=len(canonical_smi)*20+10)  # Adjust width based on SMILES length
            with open(str_svg_filename, "w", encoding='utf-8') as f:
                f.write(colored_svg)

    print(f"Generated SVGs, label PNGs, and colored SMILES SVGs for {num_molecules} molecules with {generations_per_molecule} generations each.")


from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import rdDepictor
import rdkit.Chem.Draw as rdDraw
import rdkit.Chem.AllChem as AllChem
import os
import pickle

def generate_noncolored_molecule_svgs(pkl_path: str, output_folder: str) -> None:
    """Draw the target structures WITHOUT probability colours (plain reference images).

    Args:
        pkl_path: result pickle that contains the key 'results_dict_ZINC_greedy_bl'.
        output_folder: destination directory.

    Why it exists: the paper places a plain structure next to the coloured ones,
    so the reader can see the molecule before looking at the attribution colours.

    NOTE (bugs, see header): `idx` is undefined (loop variable is `smile_idx`),
    and the trailing print(output_path) is outside the loop.
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # Load pickle file
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)

        # NOTE: everything below is indented inside the `with` block -- valid Python,
        # but it means the file handle stays open while all SVGs are written.
        # Get SMILES from trg_conv_SMI_list
        # Different key from load_data()'s 'results_dict_bl_ZINC':
        # this is the greedy-sample results of the ZINC validation run.
        smiles_list = data['results_dict_ZINC_greedy_bl'].get('trg_conv_SMI_list', [])

        for smile_idx, smiles in enumerate(smiles_list):
            #try:
                # Create molecule from SMILES
                mol = Chem.MolFromSmiles(smiles)
                if mol is None:
                    print(f"Could not create molecule from SMILES: {smiles}")
                    continue

                # Generate 2D coordinates with the same parameters as colored version
                rdDepictor.Compute2DCoords(mol)

                # Create drawer with the same dimensions as in the colored version
                d = rdDraw.rdMolDraw2D.MolDraw2DSVG(300, 500)

                # Draw molecule
                d.drawOptions().useBWAtomPalette()
                d.DrawMolecule(mol)
                d.FinishDrawing()

                # Get SVG text
                svg = d.GetDrawingText()

                # Add white background to match the style of colored versions
                svg = svg.replace('<svg ', '<svg style="background-color: white" ', 1)

                # Save SVG file
                # NOTE (bug): `idx` does not exist here -> NameError on the first loop
                output_path = os.path.join(output_folder, f"molecule_{idx}_target_noncolored.svg")
                with open(output_path, 'w') as f:
                    f.write(svg)

                print(f"Generated SVG for molecule {idx} at {output_path}")

            #except Exception as e:
            #    print(f"Error processing SMILES {smiles}: {str(e)}")
    # NOTE: outside the loop; undefined if `smiles_list` was empty
    print(output_path)
    print("Finished generating non-colored SVGs")

# Example usage:
# generate_noncolored_molecule_svgs(pkl_path, output_folder)


base_dir = os.getcwd()
# Batched-MMT experiment pickle (exp / ACD / Sim variants only differ by this path)
pkl_path = os.path.join(base_dir,"past_experiments/ChemXriv/8.0_Experiment_Sim_Exp/20240724_IC_batched_MMT_a1/20240731_115505_exp_sim_data_11_1.pkl")
# Same output root as cell 4 (not experiments/exp_1)
output_folder = os.path.join(base_dir,"experiments")
# Turns this cell into a runnable demo: renders 5 targets x 8 generations
visualize_molecules(pkl_path, output_folder)
# NOTE: this call raises NameError because of the `idx` bug described above
generate_noncolored_molecule_svgs(pkl_path, output_folder)

Generated SVGs, label PNGs, and colored SMILES SVGs for 5 molecules with 8 generations each.
Generated SVG for molecule 8 at /projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/___FIGURES_PAPERS/Figures_Paper_2/precomputed_raw_data/20241022_Explainability_Model_aligned_mol_7_0/molecule_8_target_noncolored.svg
/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/___FIGURES_PAPERS/Figures_Paper_2/precomputed_raw_data/20241022_Explainability_Model_aligned_mol_7_0/molecule_8_target_noncolored.svg
Finished generating non-colored SVGs


In [ ]:
# (intentionally empty code cell)
# Placeholder kept between the two "versions" above; nothing is defined or executed here.

### Test functions

Interactive scratch section: instead of running the full `visualize_molecules()`
pipeline, these cells let you poke at a single pickle by hand.

| Cell | What it does |
| --- | --- |
| 9 | empty placeholder |
| 10 | sets `pkl_path` to the batched-MMT experiment pickle |
| 11 | `result = load_data(pkl_path)` -- reuses the loader from cells 1/4 |
| 12 | types `result` to render the dict; use it to inspect the generation tuple layout (`[0]` SMILES, `[3]` probabilities, `[-3]`/`[4]` Tanimoto, `[-2]`/`[5]` HSQC/COSY errors) |
| 13 | *unrelated* demo: draws three dummy molecules, saves them as PNG with and without "alignment" |
| 14 | empty placeholder |

**Why cell 13 is here at all:** the paper figure needs the *same* target drawn a second
time in a plain style, side by side with the coloured one. Cell 13 explores that
depiction step on three hand-written SMILES, so you can iterate quickly without
touching a real experiment pickle.

**Caveat on the word "alignment":** `align_molecules()` only canonicalises each molecule
and calls `rdDepictor.Compute2DCoords()`. RDKit's default 2D layout picks an arbitrary
orientation per molecule, so two calls do *not* share a common atom frame. Real
template alignment needs
`rdDepictor.GenerateDepictionMatching2DStructure(mol, template)` (or a core/template
SMARTS), which this cell does not use.

In [ ]:
# (intentionally empty code cell)
# Placeholder; the interactive test starts in the next cell.

In [33]:
# Point the loader at one batched-MMT experiment pickle.
# `base_dir` comes from cell 4 or 6 (os.getcwd()); in a fresh kernel run `import os`
# and define base_dir = os.getcwd() first.
pkl_path = os.path.join(base_dir,"past_experiments/ChemXriv/8.0_Experiment_Sim_Exp/20240724_IC_batched_MMT_a1/20240731_115505_exp_sim_data_11_1.pkl")

In [34]:
# Load the results dict (target SMILES -> batches -> generations) into memory.
# Uses load_data() from cell 1/4, which expects the key 'results_dict_bl_ZINC'
# inside the pickle -- a different family of pickles uses other keys (e.g. cell 6's
# 'results_dict_ZINC_greedy_bl').
result = load_data(pkl_path)

In [35]:
# Render the dict to inspect the generation tuple layout, e.g.
#   result['<target smiles>'][0][0] -> (smiles, ..., token probs, ..., metrics, ...)
# Use this to confirm which indices the pickle uses before writing new plot code.
result

{'CCN(CC)CCNC(=O)c1cc(Br)c(N)cc1OC': [[['CCN(CC)CCNC(=O)c1cc(Br)c(N)cc1OC',
    2.760256767272949,
    45.997840881347656,
    tensor([0.9973, 0.9895, 0.9895, 0.9999, 0.9952, 0.9925, 0.9966, 0.9990, 0.9855,
            0.9772, 0.9993, 0.9990, 0.9991, 1.0000, 0.9992, 1.0000, 0.9906, 0.9955,
            0.9984, 0.0464, 1.0000, 0.9974, 0.9576, 0.9407, 0.9564, 0.9992, 0.9567,
            1.0000, 0.9579, 0.9966, 0.9980], device='cuda:0'),
    1.0,
    [0.0019983669366186213, 0.0016715491873415774]],
   ['CCN(CC)CCNC(=O)c1cc(N)c(Br)cc1OC',
    2.8225786685943604,
    46.96356964111328,
    tensor([0.9973, 0.9895, 0.9895, 0.9999, 0.9952, 0.9925, 0.9966, 0.9990, 0.9855,
            0.9772, 0.9993, 0.9990, 0.9991, 1.0000, 0.9992, 1.0000, 0.9906, 0.9955,
            0.9984, 0.8611, 0.8570, 0.9979, 0.9771, 0.0177, 1.0000, 0.9996, 0.9518,
            1.0000, 0.9553, 0.9967, 0.9982], device='cuda:0'),
    0.9069767441860465,
    [0.02128619997786351, 0.006526293427044074]],
   ['CCN(CC)CCNC(=O)c1cc

In [17]:
# =============================================================================
# CELL 13 -- Depiction demo: plain vs "aligned" images for three dummy molecules
#
# Self-contained scratch cell for the *drawing* step only -- it touches no
# experiment pickle (the pkl_path defined at the bottom is unused leftovers).
#
# Per molecule it writes two PNGs into <repo>/experiments:
#   molecule_<i>_original.png         RDKit default 2D layout
#   molecule_<i>_aligned.png          canonical SMILES + Compute2DCoords
# plus a *_label.png for each, rendered with PIL.
#
# NOTE (misleading name, left unchanged): `align_molecules()` does NOT align
#   molecules to a common frame. `rdDepictor.Compute2DCoords()` computes an
#   independent layout per molecule with an arbitrary orientation, so the three
#   images are not superimposable. True alignment needs a template, e.g.
#   rdDepictor.GenerateDepictionMatching2DStructure(mol, template).
#
# NOTE (shadowing, left unchanged): `create_label_image` is redefined here with a
#   different signature (single line, height=50) than cell 4's multi-line
#   height=120 version. Running this cell changes what later calls to
#   create_label_image(label) produce (only the first line would be visible in a
#   multi-line label pasted into the 50 px canvas... the call itself still works
#   but the layout differs).
#
# NOTE (robustness, left unchanged): the bare `except:` around font lookup also
#   catches KeyboardInterrupt and code errors; PIL's default bitmap font is then
#   used, which looks different from DejaVu Sans.
# =============================================================================

from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from rdkit.Chem import rdDepictor
import os
from PIL import Image, ImageDraw, ImageFont
import matplotlib.font_manager as fm

def create_dummy_molecules() -> list:
    """Parse the three hard-coded demo SMILES into RDKit molecules.

    The three structures are close analogues (same N-phenyl-acetamide scaffold
    with different substituents), which makes them suitable for showing whether
    a depiction is consistent across a series.

    Returns:
        List of three rdkit.Chem.Mol objects.
    """
    smiles_list = [
        'CCc1ccc(CC(=O)Nc2ccccc2)cc1',
        'CCc1ccc(CC(=O)Nc2cccc(Cl)c2)cc1',
        'CCc1ccc(CC(=O)Nc2ccc(F)cc2)cc1'
    ]
    mols = [Chem.MolFromSmiles(smiles) for smiles in smiles_list]
    return mols

def align_molecules(mols: list) -> list:
    """"Canonicalise" each molecule and compute fresh 2D coordinates.

    Args:
        mols: list of RDKit molecules.

    Returns:
        New list of molecules built from canonical SMILES with 2D coordinates.

    NOTE: this is canonicalisation + layout, NOT inter-molecule alignment -- see
    the NOTE in the cell header.
    """
    aligned_mols = []
    for mol in mols:
        # Generate a canonical SMILES
        # (MolToSmiles then CanonSmiles; the round-trip is redundant since
        #  MolToSmiles is already canonical by default)
        can_smiles = Chem.CanonSmiles(Chem.MolToSmiles(mol))
        # Create a new molecule from the canonical SMILES
        aligned_mol = Chem.MolFromSmiles(can_smiles)
        # Generate 2D coordinates
        rdDepictor.Compute2DCoords(aligned_mol)
        aligned_mols.append(aligned_mol)
    return aligned_mols

def create_label_image(label: str, width: int = 300, height: int = 50):
    """Render a SINGLE-line label into a small white PIL image.

    Args:
        label: text (no newlines expected at this height).
        width: image width in px.
        height: image height in px (50 = one 14 px line with margins).

    Returns:
        PIL.Image (RGB).
    """
    # White canvas, same width as the molecule PNG so the two stack vertically
    image = Image.new('RGB', (width, height), color='white')
    draw = ImageDraw.Draw(image)

    try:
        # Locate a TTF font through matplotlib's font manager
        font_path = fm.findfont(fm.FontProperties(family='DejaVu Sans'))
        font = ImageFont.truetype(font_path, 14)
    except:
        # Fallback: PIL's built-in bitmap font (fixed size, no scaling)
        font = ImageFont.load_default()

    draw.text((5, 5), label, fill='black', font=font)

    return image

def save_molecule_images(mols: list, output_folder: str, aligned: bool = False) -> None:
    """Save each molecule as a 300x300 PNG plus a matching label PNG.

    Args:
        mols: molecules to draw (already have 2D coordinates or will get defaults).
        output_folder: destination directory (created if missing).
        aligned: only changes the file-name suffix and the label wording
            ('aligned'/'original'); the drawing code is identical either way.
    """
    os.makedirs(output_folder, exist_ok=True)

    for i, mol in enumerate(mols):
        # Generate and save molecule image
        # MolToImage rasterises the depiction at a fixed 300x300 px
        img = Draw.MolToImage(mol, size=(300, 300))
        img_filename = os.path.join(output_folder, f"molecule_{i}_{'aligned' if aligned else 'original'}.png")
        img.save(img_filename)

        # Create and save label image
        label = f"{'Aligned' if aligned else 'Original'} Molecule {i+1}"
        label_image = create_label_image(label)
        label_filename = os.path.join(output_folder, f"molecule_{i}_{'aligned' if aligned else 'original'}_label.png")
        label_image.save(label_filename)

# Main execution
base_dir = os.getcwd()
# Unused here -- copied from the visualisation cells above
pkl_path = os.path.join(base_dir,"past_experiments/ChemXriv/8.0_Experiment_Sim_Exp/20240724_IC_batched_MMT_a1/20240731_115505_exp_sim_data_11_1.pkl")
# Same output root as cells 4/6
output_folder = os.path.join(base_dir,"experiments")

# Pass 1: the molecules exactly as parsed from SMILES
dummy_mols = create_dummy_molecules()
save_molecule_images(dummy_mols, output_folder, aligned=False)

# Pass 2: canonical SMILES + freshly computed coordinates, saved as *_aligned.png
aligned_mols = align_molecules(dummy_mols)
save_molecule_images(aligned_mols, output_folder, aligned=True)

print(f"Images saved in the '{output_folder}' directory.")

Images saved in the '/projects/cc/knlr326/1_NMR_project/1_NMR_data_AZ/___FIGURES_PAPERS/Figures_Paper_2/precomputed_raw_data/20241014_Explainability_Model' directory.


In [ ]:
# (intentionally empty code cell)
# End of the notebook; nothing left to define.